# Memoria, contexto y grafos — Notebook B: grafos (Clase 9)

*La similitud trae lo que se parece; un grafo trae lo que está conectado.*

Este notebook acompaña al momento 5 del deck, **datos conectados**: la parte «Grafos de conocimiento» de la semana 10. El deck **enuncia**; acá **exhibimos** cada mecanismo una vez, corriendo de arriba a abajo. Las secciones tienen el número de las del deck. No necesita nada del notebook A: reconstruye el corpus y el buscador por su cuenta.

0. Preparación: el entorno, el generador, el corpus de la clase de RAG y su buscador.

**Momento 5 · Datos conectados**

15. Lo que la similitud no conecta: el corpus ya era un grafo de citas.
16. Escribir el grafo: extraer entidades y relaciones con un schema, resolver entidades, auditar.
17. Recorrer el grafo: vincular la pregunta, buscar en el vecindario, plantillas y PageRank personalizado, contra los vectores.
18. Las preguntas globales: comunidades, un reporte por comunidad y map-reduce sobre los reportes.
19. La memoria como grafo: aristas con fecha, invalidar, y la pregunta de la cadena.

**Cómo está armado.** Todo libre y local: el grafo vive en **NetworkX**, en la memoria del notebook, sin servidor ni cuenta. El embedder es `BAAI/bge-m3`, el canal léxico `bm25s` y el generador `qwen3:8b` en Ollama.

**Cuánto tarda.** Con `SMOKE_TEST = True`, unos 30 minutos más las descargas; con `False`, unos 50. Lo más caro es la extracción de la sección 16: una llamada al modelo por pedazo, de 20 a 30 segundos cada una en una T4.

> **Activá la GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU.*

## 0 · Preparación

Primero miramos qué hardware tocó. La respuesta cambia los tiempos, no lo que se puede correr: casi todo funciona en CPU, sólo que mucho más lento.

In [ ]:
# Que tenemos. Si no lista ninguna GPU, activala en Entorno de ejecucion -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader || echo "sin GPU"

**Todas las instalaciones van juntas y acá arriba.** Un `pip install` que actualiza algo que Colab trae precargado puede pedir **reiniciar el entorno**; si la instalación estuviera repartida por el notebook, ese reinicio llegaría con el índice ya construido y se perdería todo. Las versiones van con número exacto, por la misma razón que en la clase de RAG: una versión de más puede romper un notebook entero.

In [ ]:
# Un solo install, con numeros exactos. Si alguna linea falla, el resto igual se instala.
!pip install -q \
  "sentence-transformers==5.7.0" \
  "bm25s==0.3.11" "PyStemmer==3.1.0" \
  "networkx==3.7" \
  "huggingface_hub>=0.23" "pyarrow>=21"

# networkx 3.7 es la primera version que trae Leiden en Python puro (seccion 18). La que
# viene en Colab es anterior.

> **Si Colab pide reiniciar el entorno, reiniciá y seguí desde acá.** No hace falta volver a correr la celda de instalación.

In [ ]:
import importlib.metadata as meta
for p in ("sentence-transformers", "transformers", "bm25s", "networkx", "huggingface_hub", "numpy"):
    try:
        print("%-24s %s" % (p, meta.version(p)))
    except Exception:
        print("%-24s (no instalado)" % p)

### 0.1 · Ollama

El generador del curso corre local: instalar, levantar el servidor en segundo plano y bajar el modelo. Si el bootstrap falla, la bandera `OLLAMA_OK` queda en falso y las celdas que necesitan el generador lo dicen y siguen: el grafo de citas de la sección 15 y las comunidades de la 18 no lo necesitan.

In [ ]:
# zstd hace falta para descomprimir el instalador de Ollama en Colab.
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"   # un solo modelo en la placa: el generador
os.environ["OLLAMA_NUM_PARALLEL"] = "1"        # un pedido a la vez: un solo cache que medir

# JAX entra de prestado con bm25s y, si inicializa su backend en la GPU, PREASIGNA EL 75%
# DE LA VRAM. Ni PyTorch ni Ollama lo ven: el generador encuentra la placa llena y se
# reparte con el procesador, sin ningun error. Tiene que estar antes de importar bm25s.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_PLATFORMS"] = "cpu"

URL_OLLAMA = "http://127.0.0.1:11434"
LOG_OLLAMA = "ollama.log"     # lo que dice el servidor: si algo se cuelga, el porque esta aca

def ollama_vivo():
    try:
        return requests.get(URL_OLLAMA + "/api/tags", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

if ollama_vivo():
    print("Ollama ya estaba corriendo.")
else:
    # Barrido previo: un runner huerfano de una corrida anterior puede seguir reservando
    # VRAM, y el servidor nuevo carga el modelo en CPU sin avisar.
    subprocess.run(["pkill", "-9", "-f", "ollama"], check=False)
    time.sleep(3)
    # start_new_session=True: sin eso, interrumpir una celda le manda el SIGINT tambien al
    # servidor, y el error aparece recien varias secciones despues.
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=open(LOG_OLLAMA, "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            print("Ollama está listo.")
            break
        time.sleep(1)
    else:
        print("El servidor tardó en levantar; volvé a correr esta celda.")

OLLAMA_OK = ollama_vivo()
print("OLLAMA_OK =", OLLAMA_OK)

In [ ]:
MODEL = "qwen3:8b"     # el generador del curso. El tag va siempre explicito: sin tag es "latest"

if OLLAMA_OK:
    print("pull", MODEL, "...", flush=True)
    r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
    if r.returncode != 0:
        print("   FALLO:", (r.stderr or r.stdout).strip()[:200])
        OLLAMA_OK = False
    else:
        !ollama list
        print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())
else:
    print("Sin Ollama: las celdas que generan van a marcarlo y seguir de largo.")

### 0.2 · Banderas y ayudantes

`SMOKE_TEST` es la convención del curso. Acá achica el corpus —514 actos en vez de 1.544, el mismo prefijo que en la clase de RAG— y la cantidad de pedazos que pasan por el extractor. Los números de las láminas salen de la corrida con `False`.

In [ ]:
import re, json, random, datetime
import numpy as np

SMOKE_TEST = False     # True = 514 actos (~20 min). False = 1.544, la corrida de las laminas (~35 min).

N_ACTOS       = 514 if SMOKE_TEST else 1544
TAM_PEDAZO    = 350
SOLAPE        = 50
K             = 5
N_CANDIDATOS  = 50
NUM_CTX       = 8192
MAX_PEDAZOS   = 30 if SMOKE_TEST else 80   # pedazos que pasan por el extractor (seccion 16)
N_COMUNIDADES = 10                          # comunidades que se resumen (seccion 18)

random.seed(1337)
np.random.seed(1337)

RESULTADOS = {}
FALLAS_GENERADOR = []

print("modo:", "SMOKE_TEST (barato)" if SMOKE_TEST else "COMPLETO", "·", N_ACTOS, "actos")

**Una sola función para hablar con el generador.** `chat()` manda una lista de mensajes a la API de chat de Ollama y devuelve, además de la respuesta, **los contadores del servidor**: cuántos tokens leyó del prompt, cuánto tardó en leerlos (el *prefill*), cuántos escribió y cuánto tardó en escribirlos (el *decode*). Son los mismos contadores del notebook A, donde miden el costo de la conversación larga.

Tres decisiones, cortas. Temperatura cero, para que la corrida se pueda repetir. `think=False`: Qwen3 es un modelo híbrido, y su razonamiento no es el tema de hoy. Y **nunca una excepción**: si una llamada falla, queda una marca en la salida y el notebook sigue. Un rechazo del servidor tampoco es una excepción: es un dato, y la sección 4 lo usa.

In [ ]:
TIMEOUT_GEN = 300    # segundos por llamada. La mas larga del notebook tarda menos de un minuto.

def asegurar_ollama(forzar=False):
    """Si el servidor se murio en el medio, lo levanta otra vez. Con forzar=True lo reinicia
    aunque conteste: un servidor colgado sigue respondiendo /api/tags y no genera nada."""
    if not OLLAMA_OK or (ollama_vivo() and not forzar):
        return True
    if forzar:
        try:
            ultimas = open(LOG_OLLAMA, errors="replace").read().splitlines()[-3:]
            print("Ollama no termina de generar. Lo último de su log:")
            for l in ultimas:
                print("   " + l[:150])
        except OSError:
            pass
        subprocess.run(["pkill", "-9", "-f", "ollama"], check=False)
        time.sleep(3)
    print("Lo vuelvo a levantar...")
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=open(LOG_OLLAMA, "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            return True
        time.sleep(1)
    print("No pude levantarlo.")
    return False

def chat(mensajes, fmt=None, num_ctx=NUM_CTX, num_predict=600, temperature=0.0,
         herramientas=None, extra=None):
    """Una llamada a /api/chat. Devuelve la respuesta Y los contadores del servidor.

    Recibe:
      mensajes      la lista [{role, content}, ...], tal cual se manda
      fmt           un JSON schema para structured outputs, o None
      num_ctx       la ventana que pedimos. None NO la pide, y el servidor usa la suya
      num_predict   tope de tokens de la respuesta
      herramientas  tools para function calling (seccion 20)
      extra         campos sueltos del pedido, como truncate o shift (seccion 4)

    Devuelve {texto, mensaje, prompt, prefill_s, salida, decode_s, error}:
      prompt     tokens del prompt SEGUN EL SERVIDOR, incluidos los que vinieron del cache
      prefill_s  segundos leyendo el prompt: solo lo que NO estaba en cache
      salida     tokens generados, y decode_s los segundos que tardo en generarlos
      error      None, o el texto del rechazo si el servidor no acepto el pedido
    """
    vacio = {"texto": "", "mensaje": {}, "prompt": 0, "prefill_s": 0.0,
             "salida": 0, "decode_s": 0.0, "error": None}
    if not OLLAMA_OK:
        return dict(vacio, texto="[sin Ollama: esta celda no corrió]", error="sin Ollama")
    opciones = {"temperature": temperature, "num_predict": num_predict}
    if num_ctx:
        opciones["num_ctx"] = num_ctx
    cuerpo = {"model": MODEL, "messages": mensajes, "stream": False, "think": False,
              "options": opciones}
    if fmt:
        cuerpo["format"] = fmt
    if herramientas:
        cuerpo["tools"] = herramientas
    if extra:
        cuerpo.update(extra)
    for intento in (1, 2):
        try:
            r = requests.post(URL_OLLAMA + "/api/chat", json=cuerpo, timeout=TIMEOUT_GEN)
        except requests.exceptions.RequestException as e:
            FALLAS_GENERADOR.append("%s: %s" % (type(e).__name__, str(e)[:80]))
            if intento == 1:
                # Un timeout es un servidor colgado: se reinicia aunque conteste.
                asegurar_ollama(forzar=isinstance(e, requests.exceptions.Timeout))
                continue
            print("   ⚠ el generador no respondió (%s). Sigo." % type(e).__name__)
            return dict(vacio, texto="[el generador no respondió]", error=type(e).__name__)
        if r.status_code != 200:
            return dict(vacio, error="HTTP %d · %s" % (r.status_code, r.text[:400]))
        j = r.json()
        msg = j.get("message") or {}
        texto = msg.get("content") or ""
        # Qwen3 a veces filtra el cierre de su razonamiento aun con think=False.
        if "</think>" in texto:
            texto = texto.split("</think>")[-1]
        return {"texto": texto.strip(), "mensaje": msg,
                "prompt": j.get("prompt_eval_count", 0),
                "prefill_s": j.get("prompt_eval_duration", 0) / 1e9,
                "salida": j.get("eval_count", 0),
                "decode_s": j.get("eval_duration", 0) / 1e9, "error": None}
    return dict(vacio, texto="[el generador no respondió]", error="reintentos")

def ask(prompt, system=None, **kw):
    """Una pregunta suelta, sin historial. Devuelve solo el texto."""
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    return chat(msgs, **kw)["texto"]

def estructurado(mensajes, esquema, num_predict=600, **kw):
    """Structured output con un modelo de pydantic. Devuelve (objeto o None, llamada cruda)."""
    r = chat(mensajes, fmt=esquema.model_json_schema(), num_predict=num_predict, **kw)
    try:
        return esquema.model_validate_json(r["texto"]), r
    except Exception:
        return None, r

if OLLAMA_OK:
    print(ask("Respondé en una sola palabra: ¿capital de Francia?"))

**Cuatro ayudantes de impresión**, los mismos de la clase de RAG: `tabla()` para los resultados, `mostrar()` para un bloque con título, `parrafo()` para leer un texto largo sin scrollear de lado, y `recortar()` para que cada ítem de una lista ocupe un renglón.

In [ ]:
import textwrap

def tabla(titulo, columnas, filas):
    """Imprime una tabla chiquita. filas = [(nombre, {columna: valor}), ...]."""
    w0 = max([len(str(f[0])) for f in filas] + [10]) + 2
    anchos = [max([len(str(f[1].get(c, "—"))) for f in filas] + [len(str(c))]) + 3
              for c in columnas]
    print("\n" + titulo)
    print("─" * (w0 + sum(anchos)))
    print("".ljust(w0) + "".join(str(c).rjust(w) for c, w in zip(columnas, anchos)))
    for nombre, vals in filas:
        print(str(nombre).ljust(w0) +
              "".join(str(vals.get(c, "—")).rjust(w) for c, w in zip(columnas, anchos)))
    print()

def parrafo(texto, ancho=74, sangria="   ", lineas=None):
    """Un texto largo, envuelto en renglones cortos. Respeta los saltos que ya traiga."""
    renglones = []
    for original in re.sub(r"[ \t]+", " ", str(texto).strip()).split("\n"):
        renglones += textwrap.wrap(original, width=ancho) or [""]
    if lineas and len(renglones) > lineas:
        renglones = renglones[:lineas]
        renglones[-1] += " …"
    return "\n".join(sangria + r for r in renglones)

def mostrar(titulo, cuerpo):
    """Un bloque con titulo, para los antes-y-despues."""
    print("─" * 74)
    print(titulo)
    print("─" * 74)
    print(parrafo(cuerpo, sangria=""))
    print()

def recortar(texto, n=160):
    """Una linea, sin saltos, para meter un texto adentro de una lista."""
    limpio = re.sub(r"\s+", " ", str(texto)).strip()
    return limpio[:n] + ("…" if len(limpio) > n else "")

**Y los ayudantes de la placa.** En una T4 conviven dos runtimes que no se ven entre sí: PyTorch, que carga el embedder, y Ollama, que carga el generador. `vram()` dice quién tiene la memoria, `liberar_gpu()` devuelve la que PyTorch retiene, y `donde_corre()` confirma que el generador entró en la GPU y no en el procesador. `modelo_en_placa()` le pregunta al servidor cuánto ocupa el modelo cargado y con qué ventana: la sección 3 lo usa para medir el KV cache.

In [ ]:
import torch

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

def vram(etiqueta=""):
    """Libre en la placa, reservado por PyTorch y atribuido a este proceso."""
    if not torch.cuda.is_available():
        return
    libre = subprocess.run(["nvidia-smi", "--query-gpu=memory.free",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout.strip()
    print("   [vram] %-26s libre %6s MiB · PyTorch %5.1f GB"
          % (etiqueta, libre, torch.cuda.memory_reserved() / 1e9))

def liberar_gpu(modelo=None):
    """Devuelve la VRAM que PyTorch retiene y, si se pide, descarga el modelo de Ollama."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if modelo and OLLAMA_OK:
        subprocess.run(["ollama", "stop", modelo], check=False, capture_output=True)

def donde_corre():
    """La columna que importa es PROCESSOR: '100% GPU' es lo esperado."""
    r = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    print(r.stdout.rstrip() if r.returncode == 0 else "OLLAMA NO RESPONDE")

def modelo_en_placa():
    """Lo que el servidor dice del modelo cargado: bytes totales, bytes en GPU y ventana."""
    try:
        for m in requests.get(URL_OLLAMA + "/api/ps", timeout=5).json().get("models", []):
            if m.get("name", "").startswith(MODEL) or m.get("model", "").startswith(MODEL):
                return {"total": m.get("size", 0), "gpu": m.get("size_vram", 0),
                        "ventana": m.get("context_length")}
    except Exception:
        pass
    return None

print("ayudantes de placa listos · dispositivo:", DISPOSITIVO)

### 0.3 · Contar en tokens del generador

El costo de escribir un grafo se mide en llamadas y en tokens del modelo que lee. Bajamos el tokenizer de Qwen3 de HuggingFace —el mismo vocabulario que usa Ollama— para contar lo que mandamos.

In [ ]:
from transformers import AutoTokenizer
import transformers
transformers.logging.set_verbosity_error()

TOK_GEN = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")   # solo el tokenizer: ~11 MB

def contar(texto):
    """Tokens de un texto suelto, con el tokenizer del generador."""
    return len(TOK_GEN.encode(texto, add_special_tokens=False))

def plantilla(mensajes, generar=True):
    """La lista de mensajes convertida en el UNICO texto que el modelo lee."""
    return TOK_GEN.apply_chat_template(mensajes, tokenize=False,
                                       add_generation_prompt=generar,
                                       enable_thinking=False)

def contar_prompt(mensajes):
    """Los tokens del prompt como los cuenta el servidor: misma plantilla, mismo vocabulario,
    y el ' /no_think' que Ollama agrega al ultimo turno del usuario cuando think=False."""
    msgs = [dict(m) for m in mensajes]
    for m in reversed(msgs):
        if m["role"] == "user":
            m["content"] = m["content"] + " /no_think"
            break
    return contar(plantilla(msgs))

print("«¿Y qué dice sobre el bacalao?» son", contar("¿Y qué dice sobre el bacalao?"), "tokens del generador")

### 0.4 · El corpus y su buscador

El mismo corpus de la clase de RAG y el mismo buscador híbrido —denso más léxico, fusionados por puesto—: son la línea de base contra la que se mide el grafo. Si la clase de RAG está fresca, alcanza con correr estas celdas.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

# El mismo corpus de la clase de RAG, reconstruido con los mismos parametros.
REPO_HF, COMMIT_HF = "coastalcph/multi_eurlex", "6278c7994cd86297e823becad7fdf897faaea174"
PISO, TECHO, MIN_ART, PRESUPUESTO_CHARS, SEMILLA = 8_000, 60_000, 3, 40_000_000, 1337
RE_ARTICULO = re.compile(r"^[ \t]*Art[ií]culo\s+\d+", re.MULTILINE | re.IGNORECASE)

def reconstruir_corpus():
    marcos = []
    for split in ("validation", "test"):
        for rev in (COMMIT_HF, "refs/convert/parquet"):
            try:
                marcos.append(pd.read_parquet(hf_hub_download(
                    REPO_HF, "es/%s/0000.parquet" % split, repo_type="dataset", revision=rev)))
                break
            except Exception:
                continue
    if not marcos:
        raise RuntimeError("No se pudo bajar el corpus del espejo parquet.")
    d = pd.concat(marcos, ignore_index=True)
    d["largo"] = d["text"].str.len()
    d["articulos"] = d["text"].map(lambda t: len(RE_ARTICULO.findall(t)))
    d = d[(d.largo >= PISO) & (d.largo <= TECHO) & (d.articulos >= MIN_ART)]
    d = d.sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)
    return d[d["largo"].cumsum() <= PRESUPUESTO_CHARS].reset_index(drop=True)

corpus = reconstruir_corpus().head(N_ACTOS).reset_index(drop=True)
corpus["celex_id"] = corpus["celex_id"].astype(str)
TEXTO_DE = dict(zip(corpus.celex_id, corpus.text))

RE_FECHA = re.compile(r"^de\s+\d{1,2}\s+de\s+\w+", re.IGNORECASE)

def titulo_informativo(texto, tope=160):
    """El tipo de acto MAS su clausula descriptiva, salteando la linea de la fecha."""
    lineas = [l for l in (re.sub(r"[ \t]+", " ", l).strip() for l in texto.split("\n")) if l]
    if not lineas:
        return ""
    tema = next((l for l in lineas[1:6] if len(l) >= 40 and not RE_FECHA.match(l)), "")
    return (lineas[0] + (" — " + tema if tema else ""))[:tope]

TITULO_DE = {c: titulo_informativo(t) for c, t in TEXTO_DE.items()}

print("actos      :", len(corpus))
print("caracteres : %.1f M" % (corpus.largo.sum() / 1e6))
print("primero    :", corpus.celex_id.iloc[0], "— el mismo que en la clase de RAG")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3", device=DISPOSITIVO)
if DISPOSITIVO == "cuda":
    embedder.half()
# El mismo tope que en RAG: bge-m3 declara 8.192 tokens, el pipeline usa 1.024. Todo lo que
# pase de ahi se corta EN SILENCIO, por la derecha. La seccion 3 lo vuelve a necesitar.
embedder.max_seq_length = 1024
TOK_EMB = embedder.tokenizer

def embeber(textos, lote=16):
    return embedder.encode(list(textos), batch_size=lote, normalize_embeddings=True,
                           show_progress_bar=len(textos) > 400).astype(np.float32)

def contar_emb(texto):
    """Tokens de un texto segun el EMBEDDER. No es la misma unidad que contar()."""
    return len(TOK_EMB.encode(texto, add_special_tokens=False))

embeber(["prueba"])
vram("con el embedder cargado")

In [ ]:
def partir_en_articulos(texto):
    """[(etiqueta, cuerpo), ...] cortando por 'Articulo N' en linea propia."""
    cortes = [m.start() for m in RE_ARTICULO.finditer(texto)]
    if not cortes:
        return [("encabezado", texto)]
    partes = [("encabezado", texto[:cortes[0]])] if cortes[0] > 0 else []
    for i, ini in enumerate(cortes):
        fin = cortes[i + 1] if i + 1 < len(cortes) else len(texto)
        etiqueta = re.sub(r"\s+", " ", texto[ini:fin].split("\n")[0]).strip()
        partes.append((etiqueta, texto[ini:fin]))
    return partes

def partir_por_tokens(texto, tam=TAM_PEDAZO, solape=SOLAPE):
    ids = TOK_EMB.encode(texto, add_special_tokens=False)
    pedazos, paso = [], max(1, tam - solape)
    for i in range(0, len(ids), paso):
        trozo = ids[i:i + tam]
        if len(trozo) < 20 and pedazos:
            break
        pedazos.append(TOK_EMB.decode(trozo))
    return pedazos

def pedazos_de(texto, celex):
    """Corte por estructura: un articulo que entra va entero; uno largo se parte en tokens.
    Cada pedazo lleva adelante el titulo del acto, como en la clase de RAG."""
    salida = []
    for etiqueta, cuerpo in partir_en_articulos(texto):
        trozos = [cuerpo] if contar_emb(cuerpo) <= TAM_PEDAZO else partir_por_tokens(cuerpo)
        for tr in trozos:
            seccion = "" if etiqueta == "encabezado" else " · " + etiqueta
            salida.append({"id": "%s:%d" % (celex, len(salida)), "celex": celex,
                           "seccion": etiqueta, "crudo": tr,
                           "texto": "[%s%s] %s" % (recortar(TITULO_DE[celex], 140), seccion, tr)})
    return salida

t0 = time.time()
PEDAZOS = [p for c, t in TEXTO_DE.items() for p in pedazos_de(t, c)]
print("pedazos:", len(PEDAZOS), "· cortados en %.0f s" % (time.time() - t0))

t0 = time.time()
MATRIZ = embeber([p["texto"] for p in PEDAZOS])
print("índice denso listo en %.0f s" % (time.time() - t0))
vram("con el índice")

In [ ]:
import logging
import bm25s, Stemmer
logging.getLogger("bm25s").setLevel(logging.WARNING)

# Las citas normativas sobreviven enteras a la tokenizacion: 1256/2011 -> n_1256_2011.
RE_ID_NORM = re.compile(r"(\d{1,5})/(\d{1,5})/(CEE|CE|UE|CECA|Euratom)", re.IGNORECASE)
RE_ID_NUM  = re.compile(r"(?<![^\W\d_])n\.?[oº°]?\s*(\d{1,5})/(\d{2,4})", re.IGNORECASE)

def normalizar_citas(t):
    t = RE_ID_NORM.sub(lambda m: "%s_%s_%s" % (m.group(1), m.group(2), m.group(3).upper()), t)
    return RE_ID_NUM.sub(lambda m: "n_%s_%s" % (m.group(1), m.group(2)), t)

stemmer = Stemmer.Stemmer("spanish")

def tokenizar(textos):
    return bm25s.tokenize([normalizar_citas(t) for t in textos],
                          stopwords="es", stemmer=stemmer, show_progress=False)

bm25_idx = bm25s.BM25()
bm25_idx.index(tokenizar([p["texto"] for p in PEDAZOS]), show_progress=False)

def buscar_bm25(consulta, n):
    idx, _ = bm25_idx.retrieve(tokenizar([consulta]), k=min(n, len(PEDAZOS)),
                               show_progress=False)
    return [PEDAZOS[int(i)] for i in idx[0]]

def rrf(listas, n, k=60):
    """Reciprocal Rank Fusion: cada lista vota con 1/(k + puesto)."""
    puntaje, por_id = {}, {}
    for lista in listas:
        for puesto, p in enumerate(lista, 1):
            puntaje[p["id"]] = puntaje.get(p["id"], 0.0) + 1.0 / (k + puesto)
            por_id[p["id"]] = p
    return [por_id[i] for i in sorted(puntaje, key=lambda i: -puntaje[i])[:n]]

def buscar(consulta, k=K):
    """El buscador de la clase de RAG: denso + lexico, fusionados por puesto.
    OJO: el canal denso trunca la consulta a los 1.024 tokens del embedder; el lexico, no."""
    q = embeber([consulta])[0]
    densos = [PEDAZOS[int(i)] for i in np.argsort(MATRIZ @ q)[::-1][:N_CANDIDATOS]]
    return rrf([densos, buscar_bm25(consulta, N_CANDIDATOS)], k)

def a_documentos(pasajes):
    """Los actos de los que salieron los pasajes, sin repetir y en orden de llegada."""
    vistos = []
    for p in pasajes:
        if p["celex"] not in vistos:
            vistos.append(p["celex"])
    return vistos

def mostrar_top(pasajes, gold=None, titulo=None):
    if titulo:
        print(titulo)
        print("─" * 74)
    for i, p in enumerate(pasajes, 1):
        marca = "   ← el correcto" if gold and p["celex"] == gold else ""
        print("%d. %-11s %-13s%s" % (i, p["celex"], recortar(p["seccion"], 13), marca))
        print("   " + recortar(TITULO_DE.get(p["celex"], ""), 110))
    print()

mostrar_top(buscar("posibilidades de pesca del Mar Báltico para 2012", 3),
            gold="32011R1256", titulo="prueba del buscador")

**Y el generador, conversando.** Hasta la clase de RAG, el prompt era uno: instrucciones, pasajes, pregunta. Ahora es una **lista de mensajes**: el de sistema con las instrucciones, después los turnos anteriores, y al final el turno de ahora con sus pasajes. Dos decisiones que la clase discute más adelante y que acá quedan escritas:

- La respuesta sale con el mismo schema de RAG (análisis, si alcanza el contexto, cita, respuesta). **Al historial va sólo lo que vio el usuario**: la respuesta y su cita. Ni el análisis ni los pasajes.
- Los pasajes viajan en el último mensaje, y sólo en ése. En el turno siguiente se recuperan otra vez.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

SISTEMA = ("Sos un asistente que responde preguntas sobre normativa europea, en una conversación. "
           "Respondé con los pasajes del <contexto> del último mensaje y con lo que se dijo en la "
           "conversación; no uses conocimiento propio. Si no alcanza, decilo. En cita van los "
           "identificadores entre corchetes de los pasajes que usaste. Respondé en dos o tres "
           "oraciones. Si el usuario sólo te cuenta algo, acusá recibo en una oración.")

class RespuestaChat(BaseModel):
    # Los campos van en orden alfabetico Y en el orden en que se piensan: el analisis antes
    # de la respuesta. Asi el orden es el mismo lo serialice como lo serialice el servidor.
    analisis: str = Field(description="Qué dicen el contexto y la conversación sobre la pregunta, en una oración.")
    basta_el_contexto: bool = Field(description="True si el contexto y la conversación alcanzan para responder.")
    cita: list[str] = Field(description="Identificadores entre corchetes de los pasajes usados.")
    respuesta: str = Field(description="La respuesta para el usuario.")

def armar_contexto(pasajes):
    """Cada pasaje con su identificador y, adentro, el titulo del acto adelante, como en la clase
    de RAG: un articulo suelto dice «el presente Reglamento» y no dice cual es."""
    return "\n\n".join("[%s · %s]\n%s" % (p["celex"], recortar(p["seccion"], 30), p["texto"])
                       for p in pasajes)

def mensaje_usuario(pregunta, pasajes):
    """El turno de ahora: con sus pasajes si los hay, o la pregunta sola."""
    if not pasajes:
        return pregunta
    return "<contexto>\n%s\n</contexto>\n\nPregunta: %s" % (armar_contexto(pasajes), pregunta)

def responder(pregunta, historial=(), consulta=None, pasajes=None, sistema=SISTEMA,
              num_ctx=NUM_CTX):
    """Un turno de la conversacion. consulta: con que se BUSCA (por default, la pregunta).
    pasajes=[] es un turno que no pregunta nada (el usuario cuenta algo): no se busca, y se
    contesta en texto, sin el schema de RAG, que lo empujaria a inventar un analisis."""
    if pasajes is None:
        pasajes = buscar(consulta or pregunta)
    msgs = [{"role": "system", "content": sistema}] + list(historial) + \
           [{"role": "user", "content": mensaje_usuario(pregunta, pasajes)}]
    if not pasajes:
        r = chat(msgs, num_predict=120, num_ctx=num_ctx)
        return dict(analisis="", basta_el_contexto=True, cita=[], respuesta=r["texto"] or (r["error"] or ""),
                    pasajes=[], mensajes=msgs, llamada=r)
    obj, r = estructurado(msgs, RespuestaChat, num_predict=800, num_ctx=num_ctx)
    if obj is None:
        obj = RespuestaChat(analisis="(no parseó)", basta_el_contexto=False, cita=[],
                            respuesta=solo_respuesta(None, r))
    return dict(obj.model_dump(), pasajes=pasajes, mensajes=msgs, llamada=r)

def limpiar_cortado(texto, tope=500):
    """Lo rescatado de un JSON cortado, sin la cola: un modelo chico atascado repite «} } }»
    hasta agotar los tokens, y esa cola NO puede entrar al historial (lo infla turno tras turno)."""
    texto = re.sub(r"(\s*[}\"]\s*){3,}.*$", "", texto, flags=re.S).strip()
    return recortar(texto, tope)

def solo_respuesta(obj, r):
    """El texto de la respuesta, aunque el JSON haya llegado cortado: si el modelo se quedó sin
    tokens a mitad del schema, se rescata el campo 'respuesta' en vez de imprimir el JSON crudo."""
    if obj is not None:
        return obj.respuesta
    m = re.search(r'"respuesta"\s*:\s*"((?:[^"\\]|\\.)*)', r["texto"] or "")
    if m:
        return limpiar_cortado(m.group(1))
    m = re.search(r'"analisis"\s*:\s*"((?:[^"\\]|\\.)*)', r["texto"] or "")
    if m:
        return "(JSON cortado; su análisis:) " + limpiar_cortado(m.group(1))
    return limpiar_cortado(r["texto"] or r["error"] or "", 400)

RE_CITA_ID = re.compile(r"\d{5}[A-Z]\d{4}[^\]\n]{0,40}")

def solo_citas(citas):
    """Las citas como [CELEX · seccion]. Un modelo chico a veces mete el pasaje entero en el
    campo de cita, y eso no tiene que entrar al historial."""
    salida = []
    for c in citas:
        m = RE_CITA_ID.search(c)
        salida.append("[%s]" % m.group(0).strip() if m else recortar(c, 50))
    return salida

def al_historial(historial, pregunta, salida):
    """Al historial va lo que se DIJO: la pregunta y la respuesta con su cita."""
    historial.append({"role": "user", "content": pregunta})
    cita = " ".join(solo_citas(salida["cita"]))
    historial.append({"role": "assistant",
                      "content": salida["respuesta"] + (" " + cita if cita else "")})

def texto_de(historial):
    """El historial como texto corrido, para leerlo o para buscar con el."""
    return "\n".join("%s: %s" % ("Usuario" if m["role"] == "user" else "Asistente", m["content"])
                     for m in historial)

> **Checkpoint.** Están el generador, el corpus y el buscador de siempre. De acá en adelante, el mismo corpus como grafo.

## 15 · Lo que la similitud no conecta

*Momento 5 · Datos conectados*

Algunas preguntas no se responden con el pasaje que más se parece: piden **seguir vínculos**. Antes de escribir un grafo con un modelo, una observación que estuvo ahí desde la clase de RAG: las normas europeas se citan unas a otras con un formato fijo, «Reglamento (CE) n.º 1224/2009». Una expresión regular, sin ningún modelo, convierte esas citas en aristas. El corpus ya era un grafo.

Dos detalles. Las citas se pasan a CELEX —el identificador del acto— para que «no 1224/2009» y «n.o 1224/2009» sean el mismo nodo. Y un acto citado que no está en el corpus igual es un nodo: no es un documento, es una entidad a la que apuntan documentos.

In [ ]:
import networkx as nx

RE_CITA_REG = re.compile(r"Reglamento\s+\((CEE|CE|UE)(?:,\s*Euratom)?\)\s*n\.?\s*[oº°]?\s*(\d{1,4})/(\d{2,4})",
                         re.IGNORECASE)
RE_CITA_DIR = re.compile(r"(Directiva|Decisi[oó]n)\s+(?:\((?:UE|CE)\)\s*)?(\d{2,4})/(\d{1,4})/(CEE|CE|UE)",
                         re.IGNORECASE)

def _anio(a):
    a = int(a)
    return a + 1900 if 50 < a < 100 else (a + 2000 if a < 100 else a)

def citas_de(texto):
    """{celex: (forma escrita, contexto)} de las normas que cita un texto. Reglamentos:
    numero/anio; directivas y decisiones: anio/numero. El contexto son los ~150 caracteres que
    siguen a la primera mencion: suele decir de que trata la norma citada."""
    salida = {}
    for m in RE_CITA_REG.finditer(texto):
        x = "3%04dR%04d" % (_anio(m.group(3)), int(m.group(2)))
        salida.setdefault(x, (re.sub(r"\s+", " ", m.group(0)), texto[m.end():m.end() + 160]))
    for m in RE_CITA_DIR.finditer(texto):
        tipo = "L" if m.group(1).lower().startswith("dir") else "D"
        x = "3%04d%s%04d" % (_anio(m.group(2)), tipo, int(m.group(3)))
        salida.setdefault(x, (re.sub(r"\s+", " ", m.group(0)), texto[m.end():m.end() + 160]))
    return salida

CITAS = nx.DiGraph()          # acto -> norma que cita
ETIQUETA = {}                 # un nombre legible por nodo
DESCRIPCION = {}              # lo que se sabe del nodo en texto: sirve para vincularlo (seccion 17)
for c, t in TEXTO_DE.items():
    CITAS.add_node(c, en_corpus=True)
    ETIQUETA[c] = recortar(TITULO_DE[c], 80)
    DESCRIPCION[c] = TITULO_DE[c]
    for x, (forma, contexto) in citas_de(t).items():
        if x == c:
            continue
        CITAS.add_edge(c, x, rel="cita")
        if x not in TEXTO_DE:
            ETIQUETA.setdefault(x, forma)
            DESCRIPCION.setdefault(x, forma + " " + re.sub(r"\s+", " ", contexto))

externos = [n for n in CITAS if n not in TEXTO_DE]
internas = sum(1 for a, b in CITAS.edges if b in TEXTO_DE)
RE_MODIFICA = re.compile(r"(?:modifica|se modifica|queda modificado)[^.]{0,120}?Reglamento\s+\((?:CEE|CE|UE)\)"
                         r"\s*n\.?\s*[oº°]?\s*\d{1,4}/\d{2,4}", re.IGNORECASE)
modifican = sum(1 for t in TEXTO_DE.values() if RE_MODIFICA.search(t[:3000]))
print("actos del corpus      :", len(TEXTO_DE))
print("aristas de cita       :", CITAS.number_of_edges(), "(%.1f por acto)" % (CITAS.number_of_edges() / len(TEXTO_DE)))
print("normas citadas afuera :", len(externos))
print("aristas entre actos del corpus:", internas)
print("actos que modifican un reglamento en su arranque:", modifican)
print()
tabla("Los nodos más citados", ["citado por", "¿en el corpus?"],
      [(recortar(ETIQUETA[n], 60), {"citado por": CITAS.in_degree(n), "¿en el corpus?": "sí" if n in TEXTO_DE else "no"})
       for n in sorted(CITAS, key=lambda n: -CITAS.in_degree(n))[:8]])
RESULTADOS["15 · grafo de citas"] = {"aristas": CITAS.number_of_edges(), "citados": len(externos),
                                    "internas": internas, "modifican": modifican}

### 15.1 · Una respuesta que es un conjunto

«¿Qué normas del corpus se apoyan en el reglamento de control de la pesca?» En el grafo, la respuesta es un salto: los actos que citan el Reglamento (CE) n.º 1224/2009. Con el buscador de RAG, la respuesta es lo que más se parece a la pregunta. Para que la comparación sea justa, al buscador le pedimos **tantos actos como tiene el conjunto correcto**.

In [ ]:
def docs_vectoriales(pregunta, m):
    """Los m primeros actos distintos que trae el buscador hibrido, pidiendole pasajes de mas."""
    q = embeber([pregunta])[0]
    densos = [PEDAZOS[int(i)] for i in np.argsort(MATRIZ @ q)[::-1][:400]]
    fusion = rrf([densos, buscar_bm25(pregunta, 400)], 800)
    return a_documentos(fusion)[:m]

CONTROL = "32009R1224"
PREGUNTA_CONTROL = "¿Qué normas del corpus se apoyan en el reglamento de control de la pesca?"
conjunto = set(CITAS.predecessors(CONTROL)) if CONTROL in CITAS else set()
vect = docs_vectoriales(PREGUNTA_CONTROL, max(1, len(conjunto)))
top5 = docs_vectoriales(PREGUNTA_CONTROL, K)

print("en el grafo, un salto: %d actos citan %s" % (len(conjunto), ETIQUETA.get(CONTROL, CONTROL)))
print("el buscador, con %d actos de presupuesto: %d de %d" % (len(vect), len(set(vect) & conjunto), len(conjunto)))
print()
print("EL TOP-%d DEL BUSCADOR" % K)
for c in top5:
    print("  %s %-11s %s" % ("✓" if c in conjunto else "·", c, recortar(TITULO_DE.get(c, ""), 80)))
print()
tipos = {}
for c in conjunto:
    clave = re.sub(r"\s+", " ", TEXTO_DE[c].split("\n")[0]).split(" ")[0].upper()
    tipos[clave] = tipos.get(clave, 0) + 1
print("los %d, por tipo de acto: %s" % (len(conjunto), ", ".join("%s %d" % kv for kv in sorted(tipos.items()))))
RESULTADOS["15 · el conjunto"] = {"conjunto": len(conjunto), "top-5 dentro del conjunto": len(set(top5) & conjunto),
                                  "con presupuesto igual al conjunto": len(set(vect) & conjunto)}

**Lo que hay que ver.** Dos fallas distintas. La respuesta no entra en k: cinco pasajes no son 34 actos. Y aun con presupuesto para todos, el buscador trae lo que **habla** de control, no lo que **cita** el reglamento de control: un reglamento de posibilidades de pesca o un programa de inspección del atún rojo se apoyan en él sin parecerse a la pregunta.

### 15.2 · Dos saltos

«¿Qué otros reglamentos aplican el mismo plan del bacalao que el del Báltico de 2012?» Del reglamento de 2012 al plan que cita, y del plan a los demás que lo citan.

In [ ]:
ACTO_2012, PLAN = "32011R1256", "32007R1098"
dos_saltos = (set(CITAS.predecessors(PLAN)) - {ACTO_2012}) if PLAN in CITAS else set()
print("camino: %s → cita → %s" % (ACTO_2012, ETIQUETA.get(PLAN, PLAN)))
print("        %s ← citan ← %s" % (ETIQUETA.get(PLAN, PLAN), ", ".join(sorted(dos_saltos))))
for c in sorted(dos_saltos):
    print("   %-11s %s" % (c, recortar(TITULO_DE.get(c, ""), 90)))
PREG_DOS = "¿Qué otros reglamentos aplican el mismo plan del bacalao que el del Báltico de 2012?"
v = docs_vectoriales(PREG_DOS, max(1, len(dos_saltos)))
print("\nel buscador, con %d actos de presupuesto: %d de %d · trajo %s"
      % (len(v), len(set(v) & dos_saltos), len(dos_saltos), " ".join(v)))

**Por qué el buscador anda mejor acá que en la pregunta anterior.** Los reglamentos del Báltico son casi idénticos entre sí, así que el parecido los encuentra. Pero no sabe que lo que los une es el plan, ni puede garantizar que estén todos: el grafo sí, porque cada uno está ahí por una arista.

▶ **Láminas 84–89** (la cadena que la similitud no recorre; una respuesta que es un conjunto; un grafo de conocimiento; los grafos del curso; el corpus ya era un grafo; tres grafos).

## 16 · Escribir el grafo

*Momento 5 · Datos conectados (sigue)*

Las citas unen normas con normas. Lo demás —qué especie regula una norma, en qué mar rige, qué medida establece— está en prosa. «Se aplicará al bacalao, eglefino, carbonero y merlán capturados en el Mar del Norte y el Skagerrak» son seis aristas que nadie escribió como cita. Las escribe un modelo: una llamada por pedazo, con un schema.

**Qué pedazos.** Los artículos de los actos de pesca que nombran una especie o una zona de pesca, hasta `MAX_PEDAZOS`. El grafo de dominio de la clase es chico a propósito: escribirlo cuesta una llamada por pedazo, y lo medimos.

In [ ]:
RE_PESCA = re.compile(r"\bpesca|pesquer|\bcapturas?\b", re.IGNORECASE)
RE_ESPECIE_ZONA = re.compile(
    r"bacalao|arenque|espadín|salmón|solla|eglefino|carbonero|merlán|atún|caballa|jurel|merluza|"
    r"cigala|anchoa|sardina|lubina|Mar del Norte|Báltico|Skagerrak|Kattegat|Mediterráneo|"
    r"Atlántico|CIEM|Mar Negro", re.IGNORECASE)

ACTOS_PESCA = [c for c in TEXTO_DE if RE_PESCA.search(TITULO_DE[c]) and "carbono" not in TITULO_DE[c].lower()]
UNIDADES = [p for p in PEDAZOS if p["celex"] in set(ACTOS_PESCA) and p["seccion"] != "encabezado"
            and RE_ESPECIE_ZONA.search(p["crudo"])][:MAX_PEDAZOS]
print("actos de pesca:", len(ACTOS_PESCA), "· pedazos que pasan por el extractor:", len(UNIDADES))

In [ ]:
class Entidad(BaseModel):
    nombre: str = Field(description="El nombre tal como aparece en el pasaje.")
    tipo: Literal["norma", "especie", "zona", "organismo", "medida"]

class Relacion(BaseModel):
    sujeto: str = Field(description="El nombre de la entidad de origen.")
    relacion: Literal["se_aplica_a", "rige_en", "establece", "modifica", "cita"]
    objeto: str = Field(description="El nombre de la entidad de destino.")

class Extraccion(BaseModel):
    entidades: list[Entidad]
    relaciones: list[Relacion]

PROMPT_EXTRAER = """Este pasaje es del acto {celex}: {titulo}.
Cuando el texto dice «el presente Reglamento», «la presente Decisión» o «la presente Directiva»,
se refiere a ese acto.

Pasaje:
{texto}

Extraé las entidades (normas, especies, zonas de pesca, organismos, medidas) y las relaciones
que el pasaje afirma entre ellas. Tipos de relación: se_aplica_a (norma → especie), rige_en
(norma → zona), establece (norma → medida), modifica (norma → norma), cita (norma → norma).
No agregues nada que el pasaje no diga. Como mucho 10 relaciones, y en entidades sólo las que
aparecen en ellas."""

RE_TRIPLE = re.compile(r'\{\s*"sujeto"\s*:\s*"((?:[^"\\]|\\.)*)"\s*,\s*"relacion"\s*:\s*"((?:[^"\\]|\\.)*)"'
                       r'\s*,\s*"objeto"\s*:\s*"((?:[^"\\]|\\.)*)"\s*\}')
RESCATADOS = []

def extraer(p, esquema=Extraccion, prompt=PROMPT_EXTRAER):
    """Una llamada por pedazo. Si el JSON llega cortado —el modelo se quedó sin tokens a mitad de
    la lista—, se rescatan los triples que alcanzaron a cerrarse: los docs de GraphRAG avisan de
    las respuestas mal formadas, y tirar el pedazo entero desperdicia la llamada."""
    msgs = [{"role": "user", "content": prompt.format(
        celex=p["celex"], titulo=recortar(TITULO_DE[p["celex"]], 140), texto=p["crudo"])}]
    obj, r = estructurado(msgs, esquema, num_predict=900)
    if obj is None and r["texto"]:
        triples = RE_TRIPLE.findall(r["texto"])
        validas = {"se_aplica_a", "rige_en", "establece", "modifica", "cita"}
        if esquema is Extraccion:
            triples = [t for t in triples if t[1] in validas]
        if triples:
            RESCATADOS.append(p["id"])
            Rel = esquema.model_fields["relaciones"].annotation.__args__[0]
            rels = [Rel(sujeto=s, relacion=rel, objeto=o) for s, rel, o in triples]
            obj = esquema(entidades=[], relaciones=rels) if esquema is Extraccion else esquema(relaciones=rels)
    return obj, r, contar(msgs[0]["content"])

ARISTAS_CRUDAS = []           # (sujeto, relacion, objeto, id del pedazo)
NOMBRES_CRUDOS = {}           # nombre tal cual -> tipo
TOKENS_EXTRAER, FALLARON = 0, 0
t0 = time.time()
for i, p in enumerate(UNIDADES):
    obj, r, n_tok = extraer(p)
    TOKENS_EXTRAER += n_tok + r["salida"]
    if obj is None:
        FALLARON += 1
        continue
    for e in obj.entidades:
        NOMBRES_CRUDOS.setdefault(e.nombre.strip(), e.tipo)
    for rel in obj.relaciones:
        ARISTAS_CRUDAS.append((rel.sujeto.strip(), rel.relacion, rel.objeto.strip(), p["id"]))
    if (i + 1) % 25 == 0:
        print("  %d pedazos · %.0f s" % (i + 1, time.time() - t0))
SEG_POR_PEDAZO = (time.time() - t0) / max(1, len(UNIDADES))
print()
print("pedazos: %d · JSON cortado, con triples rescatados: %d · sin nada que rescatar: %d"
      % (len(UNIDADES), len(RESCATADOS), FALLARON))
print("entidades distintas: %d · aristas: %d" % (len(NOMBRES_CRUDOS), len(ARISTAS_CRUDAS)))
print("%.1f s por pedazo · %d tokens del generador en total" % (SEG_POR_PEDAZO, TOKENS_EXTRAER))
print("proyección para los %d pedazos del corpus entero: %.1f horas en esta placa"
      % (len(PEDAZOS), SEG_POR_PEDAZO * len(PEDAZOS) / 3600))
RESULTADOS["16 · extracción"] = {"pedazos": len(UNIDADES), "rescatados": len(RESCATADOS), "perdidos": FALLARON,
                                 "entidades": len(NOMBRES_CRUDOS),
                                 "aristas": len(ARISTAS_CRUDAS), "s por pedazo": round(SEG_POR_PEDAZO, 1),
                                 "horas para el corpus": round(SEG_POR_PEDAZO * len(PEDAZOS) / 3600, 1)}

### 16.1 · Cuánto schema

El mismo pedazo, extraído sin schema: el modelo elige los tipos y los nombres de las relaciones. Es el pedazo del artículo 2 del reglamento de cierres en tiempo real, si está entre los elegidos.

In [ ]:
class RelacionLibre(BaseModel):
    sujeto: str
    relacion: str
    objeto: str

class ExtraccionLibre(BaseModel):
    relaciones: list[RelacionLibre]

# Sin schema de verdad: ni la lista de tipos en el prompt, ni un enum en el schema.
PROMPT_LIBRE = """Este pasaje es del acto {celex}: {titulo}.

Pasaje:
{texto}

Extraé las relaciones entre entidades que el pasaje afirma, como triples (sujeto, relación,
objeto), con los nombres de relación que te parezcan mejores."""

muestra = next((p for p in UNIDADES if p["celex"] == "32010R0724" and "eglefino" in p["crudo"]), UNIDADES[0] if UNIDADES else None)
if muestra:
    print("pedazo:", muestra["id"], "·", recortar(muestra["crudo"], 160))
    libre, _, _ = extraer(muestra, ExtraccionLibre, PROMPT_LIBRE)
    estricta, _, _ = extraer(muestra, Extraccion)
    print("\nSIN SCHEMA")
    for r_ in (libre.relaciones if libre else []):
        print("   (%s, %s, %s)" % (recortar(r_.sujeto, 30), r_.relacion, recortar(r_.objeto, 30)))
    print("\nCON SCHEMA ESTRICTO")
    for r_ in (estricta.relaciones if estricta else []):
        print("   (%s, %s, %s)" % (recortar(r_.sujeto, 30), r_.relacion, recortar(r_.objeto, 30)))
    if libre:
        print("\nnombres de relación que inventó el modelo:", sorted({r_.relacion for r_ in libre.relaciones}))

### 16.2 · Una entidad, varios nombres

La **resolución de entidades**, de lo barato a lo caro. Primero, los identificadores canónicos: toda cita a una norma va a su CELEX, y «el presente Reglamento» va al acto del que salió el pedazo. Después, normalizar: minúsculas, sin tildes, sin artículos. Por último, embeddings: dos nombres del mismo tipo con coseno muy alto se fusionan.

In [ ]:
import unicodedata

def normalizar(nombre):
    t = unicodedata.normalize("NFKD", nombre.lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    t = re.sub(r"\b(el|la|los|las|del|de la|de los|de las)\b", " ", t)
    return re.sub(r"\s+", " ", t).strip(" .,;")

RE_PRESENTE = re.compile(r"presente (reglamento|decisi[oó]n|directiva)", re.IGNORECASE)
# Nombres genericos que, dentro de un pasaje, sólo pueden ser el acto del que salió: «el Reglamento»,
# «la Decisión de Ejecución», «el artículo 7». Un artículo es parte del acto: lo llevamos al acto.
RE_GENERICO = re.compile(r"^(?:el |la )?(?:presente )?(?:reglamento|decisi[oó]n|directiva)"
                         r"(?: de ejecuci[oó]n| delegad[oa])?$|^art[ií]culo\s+\d+", re.IGNORECASE)
PEDAZO_DE = {p["id"]: p for p in UNIDADES}

def canonico(nombre, celex_origen):
    """Primer escalon: identificadores. Una cita va a su CELEX; 'el presente', 'el Reglamento' a
    secas o 'el artículo 7', al acto del que salió el pedazo."""
    if RE_PRESENTE.search(nombre) or RE_GENERICO.search(nombre.strip()):
        return celex_origen
    c = citas_de(nombre)
    if c:
        return next(iter(c))
    if re.fullmatch(r"3\d{4}[RLD]\d{4}", nombre.strip()):
        return nombre.strip()
    return None

# Escalon 1 y 2: identificadores y normalizacion
RESUELTO = {}
for s, rel, o, pid in ARISTAS_CRUDAS:
    origen = PEDAZO_DE[pid]["celex"]
    for nombre in (s, o):
        RESUELTO[(nombre, origen)] = canonico(nombre, origen) or normalizar(nombre)

# Escalon 3: embeddings, dentro de cada tipo, con un umbral alto
TIPO_DE = {}
for nombre, tipo in NOMBRES_CRUDOS.items():
    TIPO_DE[normalizar(nombre)] = tipo
sueltos = sorted({v for v in RESUELTO.values() if not re.fullmatch(r"3\d{4}[RLD]\d{4}", v)})
UMBRAL = 0.92
FUSION = {}
if sueltos:
    V = embeber(sueltos)
    for i, a in enumerate(sueltos):
        if a in FUSION:
            continue
        for j in range(i + 1, len(sueltos)):
            b = sueltos[j]
            if b not in FUSION and TIPO_DE.get(a) == TIPO_DE.get(b) and float(V[i] @ V[j]) >= UMBRAL:
                FUSION[b] = a
final = lambda v: FUSION.get(v, v)

crudos = {s for s, _, _, _ in ARISTAS_CRUDAS} | {o for _, _, o, _ in ARISTAS_CRUDAS}
tras_ids = {v for v in RESUELTO.values()}
tras_todo = {final(v) for v in RESUELTO.values()}
tabla("Nombres distintos en las aristas, escalón por escalón", ["nombres"],
      [("tal como salieron", {"nombres": len(crudos)}),
       ("+ identificadores y normalización", {"nombres": len(tras_ids)}),
       ("+ embeddings (coseno ≥ %.2f)" % UMBRAL, {"nombres": len(tras_todo)})])
for b, a in list(FUSION.items())[:8]:
    print("   «%s» → «%s»" % (b, a))
RESULTADOS["16 · resolución de entidades"] = {"crudos": len(crudos), "tras identificadores": len(tras_ids),
                                             "tras embeddings": len(tras_todo)}

**Mirá las fusiones de abajo.** Un umbral alto evita juntar cosas distintas, y aun así puede equivocarse: dos especies o dos zonas con nombres parecidos no son la misma. Bajar el umbral junta más, y junta peor. Por eso los sistemas serios dejan los casos dudosos a un modelo o a una persona.

### 16.3 · El grafo de dominio, y su auditoría

Con los nombres resueltos, las aristas forman el **grafo de dominio**. Cada arista guarda el pedazo del que salió, y eso permite auditarla: ¿el pasaje nombra el objeto de la relación? Es un control automático y barato; leer una muestra a mano sigue siendo lo serio.

In [ ]:
DOMINIO = nx.MultiDiGraph()
for s, rel, o, pid in ARISTAS_CRUDAS:
    origen = PEDAZO_DE[pid]["celex"]
    a, b = final(RESUELTO[(s, origen)]), final(RESUELTO[(o, origen)])
    if a == b:
        continue
    for n in (a, b):
        if n not in DOMINIO:
            tipo = "norma" if re.fullmatch(r"3\d{4}[RLD]\d{4}", n) else TIPO_DE.get(n, "otro")
            DOMINIO.add_node(n, tipo=tipo)
    DOMINIO.add_edge(a, b, rel=rel, pedazo=pid, objeto_crudo=o)

def respaldada(objeto_crudo, pid):
    return normalizar(objeto_crudo)[:25] in normalizar(PEDAZO_DE[pid]["crudo"])

aristas = list(DOMINIO.edges(data=True))
sin_respaldo = [(a, b, d) for a, b, d in aristas if not respaldada(d["objeto_crudo"], d["pedazo"])]
por_tipo = {}
for n, d in DOMINIO.nodes(data=True):
    por_tipo[d["tipo"]] = por_tipo.get(d["tipo"], 0) + 1
print("grafo de dominio: %d nodos · %d aristas · por tipo: %s" % (DOMINIO.number_of_nodes(), len(aristas), por_tipo))
print("auditoría automática: %d de %d aristas no tienen su objeto en el pasaje" % (len(sin_respaldo), len(aristas)))
for a, b, d in sin_respaldo[:6]:
    print("   (%s, %s, %s) · pedazo %s" % (recortar(ETIQUETA.get(a, a), 30), d["rel"], recortar(b, 30), d["pedazo"]))
RESULTADOS["16 · auditoría"] = {"aristas": len(aristas), "sin respaldo": len(sin_respaldo)}

**Cómo leerlo.** El número de arriba es una cota: una arista puede tener su objeto en el pasaje y aun así estar mal —la relación equivocada, al revés—, y otra puede faltar del todo, porque el extractor no la vio. Lo que no se extrajo, el grafo no lo puede recorrer; y lo que se extrajo mal, se recorre igual.

▶ **Láminas 90–96** (las relaciones que nadie escribió como cita; extraer con un schema; tipos de nodo y de relación; cuánto schema; una entidad, varios nombres; resolver entidades, de lo barato a lo caro; el grafo es caro y puede salir con errores).

## 17 · Recorrer el grafo

*Momento 5 · Datos conectados (sigue)*

Un grafo no recibe preguntas: recibe nodos. Para recorrerlo hay que **vincular** la pregunta con los nodos de los que habla, y después elegir cómo recorrer. Unimos los dos grafos —el de citas y el de dominio— en uno, y probamos tres formas de contestar las preguntas de conexión contra el buscador de siempre.

In [ ]:
GRAFO = nx.Graph()
GRAFO.add_edges_from(CITAS.to_undirected().edges())
GRAFO.add_edges_from((a, b) for a, b in DOMINIO.to_undirected().edges())
for n in DOMINIO.nodes:
    DESCRIPCION.setdefault(n, n)
    ETIQUETA.setdefault(n, n)

NODOS = sorted(GRAFO.nodes)
V_NODOS = embeber([DESCRIPCION.get(n, n)[:300] for n in NODOS])

def vincular(mencion, k=1, tipo=None, minimo=None):
    """Los k nodos cuya descripcion mas se parece a la mencion (entity linking). Con minimo,
    solo los que pasan ese coseno: sin umbral, siempre devuelve algo, aunque no sea nada."""
    q = embeber([mencion])[0]
    puntajes = V_NODOS @ q
    orden = np.argsort(puntajes)[::-1]
    # Un anio en la mencion es un identificador a medias: entre los 20 mas parecidos, primero los
    # actos de ese anio (el CELEX lo lleva: 3AAAA...). Lo barato antes que lo caro.
    anio = re.search(r"(?<!\d)(19[5-9]\d|20[0-4]\d)(?!\d)", mencion)
    if tipo == "norma" and anio:
        top = [int(i) for i in orden[:20]]
        orden = [i for i in top if NODOS[i][1:5] == anio.group(1)] + [i for i in top if NODOS[i][1:5] != anio.group(1)] + [int(i) for i in orden[20:]]
    salida = []
    for i in orden:
        if minimo is not None and float(puntajes[int(i)]) < minimo:
            break
        n = NODOS[int(i)]
        if tipo == "norma" and not re.fullmatch(r"3\d{4}[RLD]\d{4}", n):
            continue
        if tipo in ("especie", "zona") and DOMINIO.nodes.get(n, {}).get("tipo") != tipo:
            continue
        salida.append(n)
        if len(salida) == k:
            break
    return salida

class Menciones(BaseModel):
    menciones: list[str] = Field(description="Las normas, especies o zonas que nombra la pregunta, tal como las nombra.")

def menciones_de(pregunta):
    obj, _ = estructurado([{"role": "user", "content": "Pregunta: %s\n\nListá las normas, especies y zonas de pesca que nombra." % pregunta}],
                          Menciones, num_predict=150)
    return obj.menciones if obj else [pregunta]

for m in menciones_de(PREG_DOS):
    print("«%s» → %s" % (m, ", ".join("%s (%s)" % (n, recortar(ETIQUETA.get(n, n), 50)) for n in vincular(m, 2))))

### 17.1 · Las preguntas de conexión

Seis preguntas cuya respuesta es un **conjunto** de actos, más una de control que se responde con un solo dato. Los conjuntos correctos no se escriben de memoria: las cuatro primeras salen de leer las citas, y las dos de dominio, de los actos de pesca cuyo texto nombra a la vez la especie y la zona.

Tres formas de contestar, y en las tres el presupuesto es el mismo: tantos actos como tiene el conjunto correcto.

- **Vectores**: el buscador de RAG.
- **Plantillas**: el modelo elige una consulta fija y llena sus huecos con un schema; los huecos se vinculan a nodos y la consulta se ejecuta sobre el grafo.
- **PageRank personalizado**: se esparce peso desde los nodos que nombra la pregunta, y se toman los actos que quedan más arriba.

In [ ]:
def con_texto(*palabras):
    return {c for c in ACTOS_PESCA if all(re.search(p, TEXTO_DE[c], re.IGNORECASE) for p in palabras)}

PREGUNTAS_CONEXION = [
    ("conjunto", PREGUNTA_CONTROL, set(CITAS.predecessors(CONTROL)) if CONTROL in CITAS else set()),
    ("dos saltos", PREG_DOS, dos_saltos),
    ("conjunto", "¿Qué normas del corpus remiten al reglamento de posibilidades de pesca del Báltico de 2011?",
     set(CITAS.predecessors("32010R1124")) if "32010R1124" in CITAS else set()),
    ("conjunto", "¿Qué normas del corpus aplican el reglamento de la política pesquera común de 2002?",
     set(CITAS.predecessors("32002R2371")) if "32002R2371" in CITAS else set()),
    ("dominio", "¿Qué normas regulan la pesca de bacalao en el Mar del Norte?", con_texto("bacalao", "Mar del Norte")),
    ("dominio", "¿Qué normas fijan medidas para el atún rojo?", con_texto("atún rojo")),
    ("un dato", "¿Qué porcentaje de juveniles activa un cierre en tiempo real en el Mar del Norte?", {"32010R0724"} & set(TEXTO_DE)),
]
for tipo, p, conj in PREGUNTAS_CONEXION:
    print("%-10s %3d actos · %s" % (tipo, len(conj), p))

In [ ]:
class Plantilla(BaseModel):
    plantilla: Literal["actos_que_citan", "otros_que_citan_lo_mismo", "actos_por_especie_y_zona",
                       "actos_por_especie", "ninguna"] = Field(description=(
        "actos_que_citan: los actos que citan una norma. otros_que_citan_lo_mismo: los demás actos "
        "que citan la misma norma que un acto dado. actos_por_especie_y_zona: las normas que se aplican "
        "a una especie y rigen en una zona. actos_por_especie: las normas que se aplican a una especie. "
        "ninguna: si ninguna sirve."))
    acto: str = Field("", description="El acto de partida, para otros_que_citan_lo_mismo.")
    norma: str = Field("", description="La norma citada, para actos_que_citan y otros_que_citan_lo_mismo.")
    especie: str = ""
    zona: str = ""

def actos_relacionados(nodo, rel):
    return {a for a, b, d in DOMINIO.in_edges(nodo, data=True) if d["rel"] == rel and a in TEXTO_DE}

def ejecutar(pl):
    """Las consultas fijas, sobre el grafo. El modelo no escribe ninguna: sólo elige y llena."""
    if pl.plantilla == "actos_que_citan" and pl.norma:
        n = vincular(pl.norma, 1, "norma")[0]
        return set(CITAS.predecessors(n)) & set(TEXTO_DE) if n in CITAS else set(), n
    if pl.plantilla == "otros_que_citan_lo_mismo" and pl.acto and pl.norma:
        a, n = vincular(pl.acto, 1, "norma")[0], vincular(pl.norma, 1, "norma")[0]
        return (set(CITAS.predecessors(n)) - {a}) & set(TEXTO_DE) if n in CITAS else set(), "%s / %s" % (a, n)
    if pl.plantilla in ("actos_por_especie_y_zona", "actos_por_especie") and pl.especie:
        e = (vincular(pl.especie, 1, "especie") or [None])[0]
        actos = actos_relacionados(e, "se_aplica_a") if e else set()
        if pl.plantilla == "actos_por_especie_y_zona" and pl.zona:
            z = (vincular(pl.zona, 1, "zona") or [None])[0]
            actos &= actos_relacionados(z, "rige_en") if z else set()
            return actos, "%s / %s" % (e, z)
        return actos, str(e)
    return set(), "-"

PROMPT_PLANTILLA = """Tenés cinco consultas fijas sobre un grafo de normas europeas. Elegí la que responde
la pregunta y llená sus huecos copiando lo que nombra la pregunta; los huecos que la consulta no usa
van vacíos.

Ejemplos (otras preguntas, no la tuya):
- «¿Qué actos citan la Directiva 91/414/CEE?»
  → actos_que_citan · norma: «Directiva 91/414/CEE»
- «¿Qué normas se apoyan en el reglamento de la organización común de mercados agrícolas?»
  → actos_que_citan · norma: «reglamento de la organización común de mercados agrícolas»
- «¿Qué otros actos aplican el mismo reglamento de higiene que la Decisión 2011/207/UE?»
  → otros_que_citan_lo_mismo · acto: «Decisión 2011/207/UE» · norma: «reglamento de higiene»
- «¿Qué normas regulan la pesca de merluza en el Mediterráneo?»
  → actos_por_especie_y_zona · especie: «merluza» · zona: «Mediterráneo»
- «¿Qué normas se aplican a la caballa?»
  → actos_por_especie · especie: «caballa»
- «¿Cuántos días de pesca se permiten en el Kattegat?»
  → ninguna (pide un dato de un pasaje, no un conjunto de actos)

Pregunta: {pregunta}"""

def por_plantilla(pregunta, m):
    pl, _ = estructurado([{"role": "user", "content": PROMPT_PLANTILLA.format(pregunta=pregunta)}],
                         Plantilla, num_predict=200)
    if pl is None:
        return [], "(no parseó)"
    actos, nodos = ejecutar(pl)
    huecos = " · ".join("%s=«%s»" % (k, v) for k, v in (("acto", pl.acto), ("norma", pl.norma),
                                                         ("especie", pl.especie), ("zona", pl.zona)) if v)
    return sorted(actos)[:m], "%s(%s) → %s" % (pl.plantilla, huecos, nodos)

def pagerank_personalizado(G, semillas, alpha=0.85, vueltas=40):
    """El peso arranca en las semillas y se esparce por las aristas, vuelta tras vuelta. En cada
    vuelta, cada nodo reparte una fraccion alpha de su peso entre sus vecinos, y el resto vuelve
    a las semillas: eso es lo que lo hace 'personalizado'. Gana lo conectado por muchos caminos."""
    reinicio = {s: 1.0 / len(semillas) for s in semillas}
    peso = dict(reinicio)
    for _ in range(vueltas):
        nuevo = {s: (1 - alpha) * v for s, v in reinicio.items()}
        for n, p in peso.items():
            vecinos = list(G.neighbors(n))
            if not vecinos:
                continue
            parte = alpha * p / len(vecinos)
            for v in vecinos:
                nuevo[v] = nuevo.get(v, 0.0) + parte
        peso = nuevo
    return peso

def por_pagerank(pregunta, m):
    semillas = {n for mm in menciones_de(pregunta) for n in vincular(mm, 1)}
    if not semillas:
        return []
    pr = pagerank_personalizado(GRAFO, semillas)
    return [n for n in sorted(pr, key=lambda n: -pr[n]) if n in TEXTO_DE and n not in semillas][:m]

filas, DETALLE = [], []
for tipo, pregunta, conj in PREGUNTAS_CONEXION:
    m = max(1, len(conj))
    v = docs_vectoriales(pregunta, m)
    p, como = por_plantilla(pregunta, m)
    g = por_pagerank(pregunta, m)
    fila = {"conjunto": len(conj), "vectores": len(set(v) & conj),
            "plantilla": len(set(p) & conj), "PageRank": len(set(g) & conj)}
    filas.append((recortar(pregunta, 52), fila))
    DETALLE.append((pregunta, como))
tabla("Cuántos del conjunto correcto trae cada forma (mismo presupuesto)",
      ["conjunto", "vectores", "plantilla", "PageRank"], filas)
for pregunta, como in DETALLE:
    print("  %s\n      %s" % (recortar(pregunta, 70), como))
RESULTADOS["17 · preguntas de conexión"] = dict(filas)

**Cómo leerlo, salga como salga.** Son siete preguntas: ilustrativo, no estadístico.

- **Las de citas** son el terreno del grafo: la plantilla correcta, bien llenada, trae el conjunto entero. Si falla, mirá la columna de la derecha del detalle: casi siempre es la **vinculación** —el modelo llenó el hueco con un nombre que se vinculó al nodo equivocado—, no el recorrido.
- **Las de dominio** dependen del grafo que escribió el extractor: si la arista no se extrajo, el grafo no la tiene. Comparalo con la auditoría de la sección 16.
- **La de un dato** es el control: el buscador de siempre la contesta bien, y el grafo no agrega nada.

*Nota de laboratorio:* en producción el grafo vive en una base como Neo4j y la consulta fija es una plantilla de Cypher; acá las plantillas son funciones de Python sobre NetworkX. La idea es la misma: el modelo no escribe la consulta, la elige.

▶ **Láminas 97–102** (por dónde entra la pregunta; del nodo al vecindario; seguir el camino; recorrer todos los caminos a la vez; que el modelo escriba la consulta; lo que el grafo no puede contestar).

## 18 · Las preguntas globales

*Momento 5 · Datos conectados (sigue)*

«¿Cuáles son los grandes temas de este corpus?» no nombra ninguna entidad: no hay por dónde entrar al grafo. GraphRAG usa la estructura del grafo para saber qué resumir: detecta **comunidades**, le escribe un reporte a cada una, y contesta con un map-reduce sobre los reportes.

*Nota de laboratorio:* en NetworkX, `leiden_communities` usa por defecto otra métrica de calidad (CPM) que, con sus valores por defecto, deja cada nodo solo en su comunidad. Con `metric="modularity"` —la del paper de Leiden y la de GraphRAG— aparecen los temas. Otra decisión que un framework toma por su cuenta.

In [ ]:
NO_DIR = CITAS.to_undirected()
t0 = time.time()
COMUNIDADES = sorted(nx.community.leiden_communities(NO_DIR, seed=1337, metric="modularity"), key=len, reverse=True)
print("Leiden: %d comunidades en %.1f s · modularidad %.3f"
      % (len(COMUNIDADES), time.time() - t0, nx.community.modularity(NO_DIR, COMUNIDADES)))
filas = []
for i, com in enumerate(COMUNIDADES[:12]):
    hub = max(com, key=lambda n: NO_DIR.degree(n))
    actos = [n for n in com if n in TEXTO_DE]
    filas.append(("comunidad %d" % i, {"nodos": len(com), "actos": len(actos),
                                       "centro": recortar(DESCRIPCION.get(hub, hub), 50)}))
tabla("Las comunidades más grandes del grafo de citas", ["nodos", "actos", "centro"], filas)

# Y la de la pesca: la comunidad donde cayo el reglamento de control de la seccion 15.
i_pesca = next((i for i, c in enumerate(COMUNIDADES) if CONTROL in c), None)
if i_pesca is not None:
    com = COMUNIDADES[i_pesca]
    actos = sorted((n for n in com if n in TEXTO_DE), key=lambda n: -NO_DIR.degree(n))
    print("\nla comunidad del reglamento de control: la %d · %d nodos · %d actos · los más conectados:"
          % (i_pesca, len(com), len(actos)))
    for n in actos[:5]:
        print("   %s" % recortar(TITULO_DE[n], 90))
RESULTADOS["18 · comunidades"] = {"comunidades": len(COMUNIDADES),
                                  "tamaños": [len(c) for c in COMUNIDADES[:10]],
                                  "la de pesca": [i_pesca, len(COMUNIDADES[i_pesca])] if i_pesca is not None else None}

**Lo que hay que ver.** Nadie le dijo al algoritmo de qué trata cada acto: agrupó por quién cita a quién. Muchas comunidades son temas reconocibles; otras se forman alrededor de una norma que cita todo el mundo —un reglamento de procedimiento, por ejemplo— y juntan actos de temas distintos. Una comunidad es una estructura; el reporte es lo que le pone nombre.

### 18.1 · Un reporte por comunidad, y la pregunta global

In [ ]:
class Reporte(BaseModel):
    tema: str = Field(description="El tema de la comunidad, en pocas palabras.")
    resumen: str = Field(description="De qué tratan estos actos, en dos o tres oraciones cortas (como mucho 60 palabras).")

class Parcial(BaseModel):
    aporte: str = Field(description="Qué aporta este tema a la respuesta, en una o dos oraciones.")
    puntaje: int = Field(description="De 0 (nada) a 100 (central) cuánto sirve este tema para responder.")

PREGUNTA_GLOBAL = "¿Cuáles son los grandes temas de este corpus de normativa europea?"
REPORTES, TOK = [], {"reportes": 0, "global": 0, "rag": 0, "map-reduce": 0}
for com in COMUNIDADES[:N_COMUNIDADES]:
    actos = sorted((n for n in com if n in TEXTO_DE), key=lambda n: -NO_DIR.degree(n))[:12]
    hubs = sorted(com, key=lambda n: -NO_DIR.degree(n))[:3]
    prompt = ("Normas más citadas del grupo:\n%s\n\nActos del grupo:\n%s\n\nEscribí el reporte de este grupo de normas."
              % ("\n".join("- " + recortar(DESCRIPCION.get(h, h), 160) for h in hubs),
                 "\n".join("- " + recortar(TITULO_DE[a], 150) for a in actos)))
    obj, r = estructurado([{"role": "user", "content": prompt}], Reporte, num_predict=700)
    TOK["reportes"] += contar(prompt) + r["salida"]
    if obj:
        REPORTES.append({"tema": obj.tema, "resumen": obj.resumen, "actos": len([n for n in com if n in TEXTO_DE])})
print("reportes escritos: %d de %d" % (len(REPORTES), min(N_COMUNIDADES, len(COMUNIDADES))))
for rep in REPORTES:
    print("· %s (%d actos): %s" % (rep["tema"], rep["actos"], recortar(rep["resumen"], 120)))

# map: la pregunta a cada reporte; reduce: los aportes que más sirven
parciales = []
for rep in REPORTES:
    prompt = "Tema: %s\n%s\n\nPregunta: %s" % (rep["tema"], rep["resumen"], PREGUNTA_GLOBAL)
    obj, r = estructurado([{"role": "user", "content": prompt}], Parcial, num_predict=200)
    TOK["global"] += contar(prompt) + r["salida"]
    if obj and obj.puntaje > 0:
        parciales.append((obj.puntaje, rep["tema"], obj.aporte))
parciales.sort(reverse=True)
prompt = "Aportes parciales, del más útil al menos útil:\n%s\n\n%s Respondé en viñetas." % (
    "\n".join("- [%s] %s" % (t, a) for _, t, a in parciales[:8]), PREGUNTA_GLOBAL)
RESP_COMUNIDADES = ask(prompt, num_predict=600)
TOK["global"] += contar(prompt)
mostrar("CON LOS REPORTES DE COMUNIDADES (map-reduce sobre %d reportes)" % len(REPORTES), RESP_COMUNIDADES)

In [ ]:
# RAG: el top-5 de siempre
s_rag = responder(PREGUNTA_GLOBAL)
TOK["rag"] = contar(s_rag["mensajes"][-1]["content"])
mostrar("CON RAG (top-%d)" % K, s_rag["respuesta"])

# map-reduce sobre fichas de una muestra de actos (lo que hacíamos sin grafo)
muestra_actos = random.Random(1337).sample(list(TEXTO_DE), min(40, len(TEXTO_DE)))
partes = []
for j in range(0, len(muestra_actos), 8):
    prompt = "Actos de normativa europea:\n%s\n\nListá los temas que tratan, en viñetas cortas." % (
        "\n".join("- " + recortar(TITULO_DE[c], 150) for c in muestra_actos[j:j + 8]))
    partes.append(ask(prompt, num_predict=300))
    TOK["map-reduce"] += contar(prompt)
prompt = "Listas parciales de temas:\n%s\n\n%s Respondé en viñetas." % ("\n\n".join(partes), PREGUNTA_GLOBAL)
RESP_MR = ask(prompt, num_predict=600)
TOK["map-reduce"] += contar(prompt)
mostrar("CON MAP-REDUCE SOBRE %d ACTOS DE MUESTRA" % len(muestra_actos), RESP_MR)

actos_com = sum(len([n for n in c if n in TEXTO_DE]) for c in COMUNIDADES[:N_COMUNIDADES])
tabla("La pregunta global por tres caminos", ["actos detrás", "llamadas al escribir", "llamadas al leer", "tokens al leer"],
      [("reportes de comunidades", {"actos detrás": actos_com, "llamadas al escribir": len(REPORTES),
                                    "llamadas al leer": len(REPORTES) + 1, "tokens al leer": TOK["global"]}),
       ("RAG top-%d" % K, {"actos detrás": len(a_documentos(s_rag["pasajes"])), "llamadas al escribir": 0,
                          "llamadas al leer": 1, "tokens al leer": TOK["rag"]}),
       ("map-reduce sobre una muestra", {"actos detrás": len(muestra_actos), "llamadas al escribir": 0,
                                         "llamadas al leer": len(partes) + 1, "tokens al leer": TOK["map-reduce"]})])
RESULTADOS["18 · pregunta global"] = {"actos detrás de los reportes": actos_com, "reportes": len(REPORTES),
                                     "tokens al escribir los reportes": TOK["reportes"]}

**Cómo leerlo.** No hay juez: la columna que importa es **cuántos actos hay detrás** de cada respuesta. RAG contesta con una esquina; el map-reduce sobre una muestra, con lo que tocó en la muestra; los reportes, con las comunidades enteras, porque se escribieron antes —por eso leer es barato: el costo se pagó al escribir los reportes—. Es la pregunta madre de la clase en su forma más pura: resumir al escribir, sin saber qué se va a preguntar, o al leer, viendo la pregunta.

▶ **Láminas 103–109** (la pregunta sin entidades; comunidades; un resumen por comunidad; preguntar a cada reporte y combinar; resumir al escribir o al leer; ¿le gana a RAG?; ¿cuándo vale un grafo?).

## 19 · La memoria como grafo

*Momento 5 · Datos conectados (sigue)*

Volvemos al usuario de la parte de memoria, ahora con su memoria en un grafo. Tres sesiones con fecha; cada mensaje es un **episodio**, y de cada uno se extraen hechos que entran al grafo como aristas con dos fechas: desde cuándo y hasta cuándo valen. Cuando un hecho nuevo contradice a uno viejo, la arista vieja no se borra: se le pone fecha de fin.

Las preguntas de las sesiones se buscan en el corpus, y el acto que trae el buscador queda en el grafo como una arista «consultó». Así, la memoria del usuario y el corpus terminan en el mismo grafo.

In [ ]:
SESIONES_B = [
    ("S1", datetime.date(2026, 3, 3), [
        "Hola. Trabajo en una pesquera de Vigo y sigo las cuotas del Mar Báltico.",
        "¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?"]),
    ("S2", datetime.date(2026, 5, 12), [
        "Hoy me interesa el Mar del Norte: ¿cuándo se puede cerrar una pesquería de un día para otro?"]),
    ("S3", datetime.date(2026, 8, 25), [
        "Te actualizo: me pasé a una pesquera de Gdansk, sobre el Báltico.",
        "¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2013?"]),
]
HOY_B = datetime.date(2026, 9, 29)

class Hecho(BaseModel):
    sujeto: str = Field(description="'usuario' o el nombre de la entidad.")
    relacion: Literal["trabaja_en", "ubicada_en", "sigue"]
    objeto: str

class Hechos(BaseModel):
    hechos: list[Hecho]

MEM = nx.MultiDiGraph()
MEM.add_node("usuario", tipo="usuario")
for sid, fecha, mensajes in SESIONES_B:
    for msg in mensajes:
        if "?" in msg:                                    # una pregunta: se busca, y queda lo consultado
            acto = a_documentos(buscar(msg, K))[0]
            MEM.add_edge("usuario", acto, rel="consultó", desde=fecha, hasta=None, sesion=sid)
            continue
        obj, _ = estructurado([{"role": "user", "content": "Mensaje del usuario: %s\n\nExtraé los hechos: dónde trabaja (trabaja_en), dónde queda ese lugar (ubicada_en) y qué sigue (sigue)." % msg}],
                              Hechos, num_predict=250)
        for h in (obj.hechos if obj else []):
            s, o = normalizar(h.sujeto), normalizar(h.objeto)
            s = "usuario" if s in ("usuario", "yo", "user") else s
            if h.relacion == "trabaja_en":                  # una relacion de un solo valor vigente
                for _, viejo, k, d in list(MEM.out_edges(s, keys=True, data=True)):
                    if d["rel"] == "trabaja_en" and d["hasta"] is None and viejo != o:
                        MEM.edges[s, viejo, k]["hasta"] = fecha      # invalidar, no borrar
            MEM.add_edge(s, o, rel=h.relacion, desde=fecha, hasta=None, sesion=sid)

for a, b, d in MEM.edges(data=True):
    print("  %-14s %-10s %-34s desde %s hasta %s" % (a, d["rel"], recortar(ETIQUETA.get(b, b), 34), d["desde"], d["hasta"] or "hoy"))

In [ ]:
def vigentes(nodo, rel, fecha):
    return [b for _, b, d in MEM.out_edges(nodo, data=True)
            if d["rel"] == rel and d["desde"] <= fecha and (d["hasta"] is None or fecha < d["hasta"])]

print("¿dónde trabajo hoy?        →", vigentes("usuario", "trabaja_en", HOY_B))
print("¿dónde trabajaba en mayo?  →", vigentes("usuario", "trabaja_en", datetime.date(2026, 5, 15)))

### 19.1 · La pregunta de la cadena

«¿Qué reglamentos de los que consulté rigen donde trabajo ahora?» Por el grafo: el empleador vigente, el lugar donde queda, la zona del corpus que le corresponde, las normas que rigen ahí —por el grafo de dominio o por el de citas—, cruzadas con las consultadas. Por vectores: los registros de la memoria más parecidos a la pregunta, y el modelo contesta con eso.

In [ ]:
PREGUNTA_CADENA = "¿Qué reglamentos de los que consulté rigen donde trabajo ahora?"

# Por el grafo
empleo = vigentes("usuario", "trabaja_en", HOY_B)
lugares = set(empleo)
for _ in range(3):                                       # seguir 'ubicada_en' hasta tres saltos
    lugares |= {b for n in list(lugares) for b in vigentes(n, "ubicada_en", HOY_B)}
zonas = {z for l in lugares for z in vincular(l, 1, "zona", minimo=0.6)} if any(
    d.get("tipo") == "zona" for _, d in DOMINIO.nodes(data=True)) else set()
rigen = {a for z in zonas for a in actos_relacionados(z, "rige_en")}
consultados = {b for _, b, d in MEM.out_edges("usuario", data=True) if d["rel"] == "consultó"}
respuesta_grafo = rigen & consultados
print("empleador vigente:", empleo, "· lugares:", sorted(lugares))
print("zona del corpus  :", sorted(zonas), "· normas que rigen ahí:", len(rigen))
print("consultados      :", sorted(consultados))
print("RESPUESTA POR EL GRAFO:", sorted(respuesta_grafo) or "(vacía: mirá qué eslabón faltó arriba)")

# Por vectores: los registros de la memoria como textos sueltos
REGISTROS = []
for a, b, d in MEM.edges(data=True):
    fechas = "desde %s%s" % (d["desde"], " hasta %s" % d["hasta"] if d["hasta"] else "")
    REGISTROS.append("%s %s %s (%s)" % (a, d["rel"].replace("_", " "), ETIQUETA.get(b, b), fechas))
V_REG = embeber(REGISTROS)
q = embeber([PREGUNTA_CADENA])[0]
top = [REGISTROS[int(i)] for i in np.argsort(V_REG @ q)[::-1][:5]]
resp_vec = ask("Hoy es %s. Memoria del usuario:\n%s\n\nPregunta del usuario: %s\nRespondé con lo que dice la memoria."
               % (HOY_B, "\n".join("- " + r for r in top), PREGUNTA_CADENA), num_predict=300)
print("\nPOR VECTORES · los cinco registros más parecidos:")
for r in top:
    print("   ", recortar(r, 110))
mostrar("RESPUESTA POR VECTORES", resp_vec)
RESULTADOS["19 · la cadena"] = {"grafo": sorted(respuesta_grafo), "vectores": recortar(resp_vec, 200)}

**Cómo leerlo.** El recorrido devuelve un conjunto y su camino —empleador vigente, lugar, zona, normas—, y si falla, el camino dice qué eslabón faltó: casi siempre uno que el extractor no escribió. La memoria vectorial devuelve registros parecidos y deja que el modelo arme la cadena, sin garantía de que estén todos los eslabones ni de que use el empleador vigente. Un grafo de memoria compra conexiones y tiempo; se paga en llamadas en cada episodio, y con un modelo chico, en errores.

▶ **Láminas 110–114** (la memoria también tiene conexiones; un grafo temporal; aristas con dos relojes; la cadena, recorrida; cuánto paga un grafo de memoria).

## Cierre — lo que exhibió el notebook B

Cada mecanismo una vez, sobre un caso: ilustrativo, no estadístico. Lo que sigue se imprime desde los datos de esta corrida.

In [ ]:
print("=" * 74)
print("LO QUE MEDIMOS · %d actos · %s" % (len(corpus), MODEL))
print("=" * 74)
for clave, valor in RESULTADOS.items():
    print("\n" + clave)
    print(parrafo(recortar(json.dumps(valor, ensure_ascii=False, default=str), 600)))
if FALLAS_GENERADOR:
    print("\n⚠ %d llamada(s) al generador fallaron:" % len(FALLAS_GENERADOR))
    for f in FALLAS_GENERADOR[:5]:
        print("   ", f)
print("\nUn grafo vale cuando las preguntas son sobre conexiones o sobre el todo; para un dato")
print("suelto, gana el buscador de siempre.")